This project has been developed and run on Google Colab Notebook

In [ ]:
pip install langchain langgraph

In [ ]:
pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 21.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


In [ ]:
import os
from google.colab import userdata

In [ ]:
# Access the secret
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

# Set it as environment variable (LangChain and Google SDK use this)
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import MessagesState, StateGraph, START
from langgraph.prebuilt import create_react_agent, InjectedState
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import MemorySaver
from langchain.tools import BaseTool
from typing import Any

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
class MultiAgentState(MessagesState):
    last_active_agent: str

In [ ]:
def get_travel_recommendations(query: str) -> str:
    """
    This function takes a user query about travel recommendations and returns a string containing travel recommendations.

    Args:
        query: The user's query about travel recommendations.

    Returns:
        A string containing travel recommendations.
    """
    # Basic keyword-based recommendations
    if "beach" in query.lower():
        recommendations = "For a beach vacation, I recommend Hawaii, Bali, or the Maldives."
    elif "mountain" in query.lower():
        recommendations = "For a mountain getaway, consider the Swiss Alps, the Rockies, or the Himalayas."
    elif "city" in query.lower():
        recommendations = "For a city break, I suggest exploring New York, Paris, or Tokyo."
    else:
        recommendations = "I recommend visiting popular destinations like Hawaii, Bali, or Paris."  # Default recommendations

    return recommendations


def make_handoff_tool(agent_name: str) -> BaseTool:
    class HandoffTool(BaseTool):
        name: str = f"handoff_to_{agent_name}"  # Type annotation added
        description: str = f"Hand off the conversation to {agent_name}."  # Type annotation added

        def _run(self,   # type: ignore
                **kwargs: Any, # Type annotation modified
            ) -> str:
            # Added an indented block here
            return agent_name

        async def _arun(self,  # type: ignore
                **kwargs: Any, # Type annotation modified
            ) -> str:
            # Added an indented block here
            return agent_name

    return HandoffTool()


travel_advisor_tools = [
    get_travel_recommendations,
    make_handoff_tool(agent_name="hotel_advisor"),
]

travel_advisor = create_react_agent(
    model,
    travel_advisor_tools,
    prompt=(
        "You are a general travel expert that can recommend travel destinations "
        "use get_travel_recommendations tool to provide travel recommendations"
        "(e.g. countries, cities, etc). If you need hotel recommendations, you must call a tool make_handoff_tool and use get_hotel_recommendations tool"
        #"If you need help picking travel destinations, you must call a tool make_handoff_tool
        "You MUST include a human-readable response before transferring to another agent."
    ),
)


'''
def call_travel_advisor(state: MultiAgentState) -> Command:
    response = travel_advisor.invoke(state)
    update = {**response, "last_active_agent": "travel_advisor"}
    return Command(update=update, goto="human")
'''

def call_travel_advisor(state: MultiAgentState) -> Command:
    response = travel_advisor.invoke(state)

    update = {
        **response,
        "last_active_agent": "travel_advisor"
    }

    # Get the last message
    last_message = response["messages"][-1]

    # print(f"last_message = {last_message}\n")

    content = getattr(last_message, "content", "").lower()

    if "hotel advisor" in content:
       # print("hotel advisor in content")
       return Command(
        update=update,
        goto="hotel_advisor",
    )

    '''
    # Did the agent call the handoff tool?
    if getattr(last_message, "tool_calls", None):
        for tool_call in last_message.tool_calls:
            if tool_call["name"] == "make_handoff_tool":
                return Command(
                    update=update,
                    goto="hotel_advisor",
                )
    '''

    return Command(
        update=update,
        goto="human"
    )

/tmp/ipykernel_1283/1047027206.py:49: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  travel_advisor = create_react_agent(


In [ ]:
def get_hotel_recommendations(query: str) -> str:
    """
    This function takes a user query about hotel recommendations and returns a string containing hotel recommendations.

    Args:
        query: The user's query about hotel recommendations.

    Returns:
        A string containing hotel recommendations.
    """
    # Basic keyword-based recommendations (You'll need to improve this logic)
    if "beach" in query.lower():
        recommendations = "For beach hotels, I recommend The Ritz-Carlton, Bali or Four Seasons Maui."
    elif "city" in query.lower():
        recommendations = "For city hotels, consider The Peninsula, Hong Kong or The Savoy, London."
    else:
        recommendations = "I recommend checking out hotels like The Ritz-Carlton or Four Seasons."

    return recommendations

In [ ]:
hotel_advisor_tools = [
    get_hotel_recommendations,  # Now using the defined function
    make_handoff_tool(agent_name="travel_advisor"),
]

hotel_advisor = create_react_agent(
    model,
    hotel_advisor_tools,
    prompt=(
        "You are a hotel expert that can provide hotel recommendations for a given destination. "
        "use get_hotel_recommendations tool to provide hotel recommendations"
        "If you need help picking travel destinations, you must call a tool make_handoff_tool and use get_travel_recommendations tool"
        "You MUST include a human-readable response before transferring to another agent."
    ),
)

'''
def call_hotel_advisor(state: MultiAgentState) -> Command:
    response = hotel_advisor.invoke(state)
    update = {**response, "last_active_agent": "hotel_advisor"}
    return Command(update=update, goto="human")
'''

def call_hotel_advisor(state: MultiAgentState) -> Command:
    response = hotel_advisor.invoke(state)

    update = {
        **response,
        "last_active_agent": "hotel_advisor"
    }

    # Get the last message
    last_message = response["messages"][-1]

    # print(f"last_message = {last_message}\n")

    content = getattr(last_message, "content", "").lower()

    if "travel advisor" in content:
       # print("travel advisor in content")
       return Command(
        update=update,
        goto="travel_advisor",
    )


    '''
    # Did the agent call the handoff tool?
    if getattr(last_message, "tool_calls", None):
        for tool_call in last_message.tool_calls:
            if tool_call["name"] == "make_handoff_tool":
                return Command(
                    update=update,
                    goto="travel_advisor",
                )
    '''

    return Command(
        update=update,
        goto="human"
    )

/tmp/ipykernel_1283/663534474.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  hotel_advisor = create_react_agent(


In [ ]:
def human_node(state: MultiAgentState, config) -> Command:
    user_input = interrupt(value="Ready for user input.")
    active_agent = state["last_active_agent"]

    return Command(
        update={
            "messages": [{"role": "human", "content": user_input}]
        },
        goto=active_agent,
    )

In [ ]:
builder = StateGraph(MultiAgentState)

builder.add_node("travel_advisor", call_travel_advisor)
builder.add_node("hotel_advisor", call_hotel_advisor)
builder.add_node("human", human_node)

builder.add_edge(START, "travel_advisor")  # Initial entry point
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
import uuid

thread_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

inputs = [
    {"messages": [{"role": "user", "content": "i want best hotels in Swiss Alps"}]},
    Command(resume="could you recommend best city travel destinations and hotels?"),
    #Command(resume="could you recommend a nice hotel in one of the areas and tell me which area it is."),
    Command(resume="i like the first one. could you recommend something to do near the hotel?"),
]

for idx, user_input in enumerate(inputs):
    print(f"\n--- Conversation Turn {idx + 1} ---\n")
    print(f"User: {user_input}\n")
    for update in graph.stream(user_input, config=thread_config, stream_mode="updates"):
        for node_id, value in update.items():
            if isinstance(value, dict) and value.get("messages", []):
                last_message = value["messages"][-1]
                if isinstance(last_message, dict) or last_message.type != "ai":
                    continue
                print(f"{node_id}: {last_message.content}")


--- Conversation Turn 1 ---

User: {'messages': [{'role': 'user', 'content': 'i want best hotels in Swiss Alps'}]}

travel_advisor: I've handed off your request to a hotel advisor who will provide you with the best hotel recommendations in the Swiss Alps. You should receive the information shortly!
hotel_advisor: Here are some top hotel recommendations in the Swiss Alps:

1. **The Ritz-Carlton, Geneva** - Known for its luxurious accommodations and stunning views of the Alps.
2. **Four Seasons Hotel des Bergues** - Offers a blend of modern luxury and traditional Swiss hospitality, with breathtaking mountain views.

These hotels provide excellent amenities and are perfect for a memorable stay in the Swiss Alps. If you need more options or specific details, feel free to ask!

--- Conversation Turn 2 ---

User: Command(resume='could you recommend best city travel destinations and hotels?')

hotel_advisor: I've handed off your request to a travel advisor who will provide you with the best 